# Bank Marketing - term deposit classification
ML Assignment 2. Five classifiers on the UCI bank marketing data.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef,
                             confusion_matrix, classification_report)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

## 1. Load the data

In [ ]:
df = pd.read_csv("data/bank-full.csv", sep=";")
print(df.shape)
df.head()

In [ ]:
df.info()
print("missing values:", df.isnull().sum().sum())
print(df["y"].value_counts())
print(df["y"].value_counts(normalize=True).round(3))

## 2. Quick look at the data

In [ ]:
df.describe().T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
sns.countplot(x="y", data=df, ax=axes[0])
axes[0].set_title("Target distribution")
sns.countplot(y="job", data=df, order=df["job"].value_counts().index, ax=axes[1])
axes[1].set_title("Job")
plt.tight_layout()
plt.show()

## 3. Train / test split and preprocessing

In [ ]:
df["y"] = (df["y"] == "yes").astype(int)

X = df.drop(columns=["y"])
y = df["y"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
print(X_train.shape, X_test.shape)

# test rows with the target, used later for the streamlit upload
os.makedirs("model", exist_ok=True)
test_data = X_test.copy()
test_data["y"] = y_test.values
test_data.to_csv("test_data.csv", index=False)

In [ ]:
cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]
print(cat_cols)
print(num_cols)

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

X_train_p = preprocessor.fit_transform(X_train)
X_test_p = preprocessor.transform(X_test)
if hasattr(X_train_p, "toarray"):
    X_train_p = X_train_p.toarray()
    X_test_p = X_test_p.toarray()
print(X_train_p.shape)

joblib.dump(preprocessor, "model/preprocessor.pkl")

## 4. Train the models and evaluate

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, min_samples_leaf=20, random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(n_neighbors=7),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": SVC(kernel="rbf", C=1.0, random_state=RANDOM_STATE),
}

file_names = {
    "Logistic Regression": "logistic_regression.pkl",
    "Decision Tree": "decision_tree.pkl",
    "KNN": "knn.pkl",
    "Naive Bayes": "naive_bayes.pkl",
    "Support Vector Machine": "svm.pkl",
}

def get_scores(model, X):
    # svm has no predict_proba here, so use the decision function for auc
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    return model.decision_function(X)

In [ ]:
results = []
predictions = {}

for name, model in models.items():
    print("training", name, "...")
    model.fit(X_train_p, y_train)

    y_pred = model.predict(X_test_p)
    y_score = get_scores(model, X_test_p)
    predictions[name] = y_pred

    results.append({
        "ML Model Name": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_score),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred),
    })
    joblib.dump(model, os.path.join("model", file_names[name]))

results_df = pd.DataFrame(results).round(4)
results_df.to_csv("model_results.csv", index=False)
results_df

## 5. Confusion matrices and reports

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, (name, y_pred) in zip(axes.ravel(), predictions.items()):
    sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d",
                cmap="Blues", ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
for name, y_pred in predictions.items():
    print(name)
    print(classification_report(y_test, y_pred, target_names=["no", "yes"]))

## 6. Comparison

In [ ]:
results_df.set_index("ML Model Name").plot(kind="bar", figsize=(11, 4))
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.title("Model comparison")
plt.legend(loc="lower right", ncol=3)
plt.tight_layout()
plt.show()